# Notebook 3: Proposed FP-Growth với In-mining Pruning

Notebook này thực hiện kịch bản **Proposed (In-mining Pruning)** cho bài thực nghiệm khai phá luật kết hợp định lượng.

Ý tưởng chính:

1. Nạp danh sách giao dịch và bảng giá item từ `data_preprocessing.ipynb`.
2. Sắp xếp item theo **giá tăng dần**.
3. Xây FP-Tree theo thứ tự giá.
4. Duyệt sinh mẫu theo DFS và kiểm tra `avg(price)` ngay trong quá trình mở rộng mẫu.
5. Nếu `avg(price) > PRICE_THRESHOLD`, dừng mở rộng nhánh đó ngay.

Điểm quan trọng là DFS luôn mở rộng itemset theo thứ tự giá không giảm. Vì vậy, khi một ứng viên đã có `avg(price)` vượt ngưỡng, mọi mở rộng sau đó chỉ thêm item có giá bằng hoặc cao hơn, nên trung bình không thể giảm xuống dưới ngưỡng.

## 1. Cấu hình môi trường

Notebook chạy được trên local và Google Colab. Các tham số có thể chỉnh trực tiếp trong cell cấu hình hoặc bằng biến môi trường:

- `OUTPUT_DIR`: thư mục chứa artifact đầu vào và đầu ra.
- `MIN_SUPPORT_RATIOS`: danh sách support, ví dụ `5%,4%,3%,2%,1%`.
- `PRICE_THRESHOLD`: ngưỡng `avg(price)`, mặc định `20`.
- Các biến tên file artifact như `TRANSACTIONS_PICKLE`, `ITEM_PRICE_PICKLE`, `PROPOSED_RESULTS_PICKLE`.

In [1]:
import importlib.util
import json
import math
import os
import pickle
import subprocess
import sys
import time
from collections import Counter
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    """Cài hoặc nạp thư viện cần thiết cho cả local và Colab."""
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        bundled_packages = Path.home() / ".cache" / "codex-runtimes" / "codex-primary-runtime" / "dependencies" / "python"
        if bundled_packages.exists() and str(bundled_packages) not in sys.path:
            sys.path.insert(0, str(bundled_packages))

    if importlib.util.find_spec(import_name) is None:
        print(f"Thiếu thư viện '{import_name}'. Đang cài đặt gói '{pip_name}'...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])


ensure_package("pandas")

import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False


# =========================
# CẤU HÌNH LINH HOẠT
# =========================
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", PROJECT_ROOT / "outputs")).resolve()

ARTIFACT_FILES = {
    "transactions_pickle": os.environ.get("TRANSACTIONS_PICKLE", "transactions.pkl"),
    "item_price_pickle": os.environ.get("ITEM_PRICE_PICKLE", "item_price.pkl"),
    "preprocessing_summary_json": os.environ.get("PREPROCESSING_SUMMARY_JSON", "preprocessing_summary.json"),
    "baseline_results_pickle": os.environ.get("BASELINE_RESULTS_PICKLE", "baseline_results.pkl"),
    "proposed_results_pickle": os.environ.get("PROPOSED_RESULTS_PICKLE", "proposed_results.pkl"),
    "proposed_metrics_csv": os.environ.get("PROPOSED_METRICS_CSV", "proposed_metrics.csv"),
    "proposed_patterns_csv": os.environ.get("PROPOSED_PATTERNS_CSV", "proposed_final_patterns.csv"),
    "proposed_summary_json": os.environ.get("PROPOSED_SUMMARY_JSON", "proposed_summary.json"),
}


def parse_support_ratios(value, default_ratios):
    """Đọc danh sách support linh hoạt, ví dụ '5%,4%,3%' hoặc '0.05,0.04,0.03'."""
    if not value:
        return default_ratios
    ratios = []
    for token in value.split(","):
        token = token.strip()
        if not token:
            continue
        if token.endswith("%"):
            ratios.append(float(token[:-1]) / 100)
        else:
            number = float(token)
            ratios.append(number / 100 if number > 1 else number)
    return ratios


MIN_SUPPORT_RATIOS = parse_support_ratios(
    os.environ.get("MIN_SUPPORT_RATIOS", ""),
    default_ratios=[0.05, 0.04, 0.03, 0.02, 0.01],
)
PRICE_THRESHOLD = float(os.environ.get("PRICE_THRESHOLD", "20"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 3000)

print("Môi trường chạy:", "Google Colab" if IN_COLAB else "Local/Jupyter")
print("Thư mục làm việc:", PROJECT_ROOT)
print("Thư mục lưu kết quả:", OUTPUT_DIR)
print("Danh sách min_support:", [f"{ratio:.0%}" for ratio in MIN_SUPPORT_RATIOS])
print("Ngưỡng avg(price):", PRICE_THRESHOLD)

Môi trường chạy: Local/Jupyter
Thư mục làm việc: H:\chuong_trinh_hoc_UEH\mon_hoc_ki_6\data_mining\Lab\Data_Mining_Labs\Mining_Quantitative_Association_Rules_Lab
Thư mục lưu kết quả: H:\chuong_trinh_hoc_UEH\mon_hoc_ki_6\data_mining\Lab\Data_Mining_Labs\Mining_Quantitative_Association_Rules_Lab\outputs
Danh sách min_support: ['5%', '4%', '3%', '2%', '1%']
Ngưỡng avg(price): 20.0


## 2. Nạp artifact từ bước tiền xử lý

Notebook sử dụng các artifact đã tạo ở notebook 1:

- Danh sách giao dịch dạng pickle.
- Bảng giá đại diện của từng item.
- File tóm tắt tiền xử lý.

Nếu chạy trên Colab, hãy chạy notebook 1 trước hoặc upload các file artifact vào cùng thư mục làm việc.

In [2]:
def find_artifact(filename, output_dir=OUTPUT_DIR):
    """Tìm artifact theo tên file cấu hình trong OUTPUT_DIR, PROJECT_ROOT hoặc /content."""
    direct_candidates = [output_dir / filename, PROJECT_ROOT / filename]
    search_roots = [PROJECT_ROOT]
    if IN_COLAB:
        search_roots.append(Path("/content"))

    seen = set()
    for path in direct_candidates:
        resolved = path.resolve()
        if resolved not in seen:
            seen.add(resolved)
            if resolved.exists():
                return resolved

    for root in search_roots:
        if not root.exists():
            continue
        for path in root.glob(f"**/{filename}"):
            resolved = path.resolve()
            if resolved not in seen:
                seen.add(resolved)
                if resolved.exists():
                    return resolved

    return output_dir / filename


transactions_path = find_artifact(ARTIFACT_FILES["transactions_pickle"])
item_price_path = find_artifact(ARTIFACT_FILES["item_price_pickle"])
summary_path = find_artifact(ARTIFACT_FILES["preprocessing_summary_json"])

required_artifacts = [transactions_path, item_price_path, summary_path]
missing_artifacts = [str(path) for path in required_artifacts if not path.exists()]
if missing_artifacts:
    raise FileNotFoundError(
        "Thiếu artifact từ bước tiền xử lý. Hãy chạy data_preprocessing.ipynb trước. "
        f"Các file còn thiếu: {missing_artifacts}"
    )

with transactions_path.open("rb") as f:
    transactions = pickle.load(f)

with item_price_path.open("rb") as f:
    item_price = pickle.load(f)

with summary_path.open("r", encoding="utf-8") as f:
    preprocessing_summary = json.load(f)

item_price = {item: float(price) for item, price in item_price.items()}
N_TRANSACTIONS = len(transactions)

load_summary_df = pd.DataFrame([
    {"Thông tin": "Số giao dịch", "Giá trị": N_TRANSACTIONS},
    {"Thông tin": "Số item có giá đại diện", "Giá trị": len(item_price)},
    {"Thông tin": "Số dòng sau tiền xử lý", "Giá trị": preprocessing_summary.get("clean_rows")},
    {"Thông tin": "Cách lấy giá đại diện", "Giá trị": preprocessing_summary.get("price_agg_method", "median")},
])

print("Đã nạp dữ liệu tiền xử lý thành công.")
display(load_summary_df)

print("Ví dụ 5 giao dịch đầu:")
display(pd.DataFrame({
    "Mã thứ tự giao dịch": list(range(1, 6)),
    "Số item": [len(t) for t in transactions[:5]],
    "Items": [" | ".join(t) for t in transactions[:5]],
}))

Đã nạp dữ liệu tiền xử lý thành công.


,Thông tin,Giá trị
0,Số giao dịch,19559
1,Số item có giá đại diện,4006
2,Số dòng sau tiền xử lý,519551
3,Cách lấy giá đại diện,median


Ví dụ 5 giao dịch đầu:


,Mã thứ tự giao dịch,Số item,Items
0,1,7,CREAM CUPID HEARTS COAT HANGER | GLASS STAR FROSTED T-LIGHT HOLDER | KNITTED UNION FLAG HOT WATER BOTTLE | RED WOOLLY HOTTIE WHITE HEART. | SET 7 BABUSHKA NESTING BOXES | WHITE HANGING HEART T-LIGHT HOLDER | WHITE METAL LANTERN
1,2,2,HAND WARMER RED POLKA DOT | HAND WARMER UNION JACK
2,3,12,ASSORTED COLOUR BIRD ORNAMENT | BOX OF 6 ASSORTED COLOUR TEASPOONS | BOX OF VINTAGE ALPHABET BLOCKS | BOX OF VINTAGE JIGSAW BLOCKS | DOORMAT NEW ENGLAND | FELTCRAFT PRINCESS CHARLOTTE DOLL | HOME BUILDING BLOCK WORD | IVORY KNITTED MUG COSY | LOVE BUILDING BLOCK WORD | POPPY'S PLAYHOUSE BEDROOM | POPPY'S PLAYHOUSE KITCHEN | RECIPE BOX WITH METAL HEART
3,4,4,BLUE COAT RACK PARIS FASHION | JAM MAKING SET WITH JARS | RED COAT RACK PARIS FASHION | YELLOW COAT RACK PARIS FASHION
4,5,1,BATH BUILDING BLOCK WORD


## 3. Cấu hình thực nghiệm Proposed

Các ngưỡng support được đổi sang support count bằng `ceil(min_support_ratio * số_giao_dịch)`. Ngưỡng định lượng mặc định là `avg(price) <= 20`.

Khác với Baseline, Proposed không sinh toàn bộ frequent itemsets rồi mới lọc. Mỗi candidate được kiểm tra giá ngay khi đang được sinh trong DFS.

In [3]:
support_config_df = pd.DataFrame([
    {
        "Min Support": f"{ratio:.0%}",
        "Tỷ lệ": ratio,
        "Support count tối thiểu": math.ceil(ratio * N_TRANSACTIONS),
    }
    for ratio in MIN_SUPPORT_RATIOS
])

print("Cấu hình thực nghiệm Proposed:")
display(support_config_df)
print(f"Ngưỡng cắt tỉa trong quá trình mining: avg(price) <= {PRICE_THRESHOLD}")

Cấu hình thực nghiệm Proposed:


,Min Support,Tỷ lệ,Support count tối thiểu
0,5%,0.05,978
1,4%,0.04,783
2,3%,0.03,587
3,2%,0.02,392
4,1%,0.01,196


Ngưỡng cắt tỉa trong quá trình mining: avg(price) <= 20.0


## 4. Cài đặt FP-Tree theo thứ tự giá và DFS pruning

Cài đặt gồm hai phần:

- **Price-ordered FP-Tree:** mỗi giao dịch được lọc item thường xuyên rồi sắp theo giá tăng dần trước khi đưa vào cây.
- **DFS in-mining pruning:** itemset được mở rộng theo đúng thứ tự giá tăng dần. Nếu `avg(price)` của candidate vượt ngưỡng, notebook dừng phần còn lại của nhánh hiện tại.

Để tính support nhanh trong DFS, notebook tạo chỉ mục TID dạng bitset cho từng item. Đây là header index dùng chung cho các ngưỡng support, giúp thao tác giao tập giao dịch được thực hiện bằng phép toán bit nhanh thay vì quét lại toàn bộ dữ liệu nhiều lần.

In [4]:
class PriceFPNode:
    """Một node trong FP-Tree sắp theo giá."""

    __slots__ = ("item", "count", "parent", "children", "link")

    def __init__(self, item=None, count=0, parent=None):
        self.item = item
        self.count = count
        self.parent = parent
        self.children = {}
        self.link = None


class PriceOrderedFPTree:
    """FP-Tree được xây theo thứ tự giá tăng dần."""

    __slots__ = ("root", "headers", "supports")

    def __init__(self):
        self.root = PriceFPNode()
        self.headers = {}
        self.supports = {}

    def add_transaction(self, ordered_items):
        current = self.root
        for item in ordered_items:
            child = current.children.get(item)
            if child is None:
                child = PriceFPNode(item=item, count=0, parent=current)
                current.children[item] = child
                child.link = self.headers.get(item)
                self.headers[item] = child
            child.count += 1
            current = child


def count_tree_nodes(tree):
    """Đếm số node thật trong FP-Tree, không tính root."""
    node_count = 0
    stack = list(tree.root.children.values())
    while stack:
        node = stack.pop()
        node_count += 1
        stack.extend(node.children.values())
    return node_count


def max_tree_depth(tree):
    """Tính độ sâu lớn nhất của FP-Tree."""
    max_depth = 0
    stack = [(child, 1) for child in tree.root.children.values()]
    while stack:
        node, depth = stack.pop()
        max_depth = max(max_depth, depth)
        stack.extend((child, depth + 1) for child in node.children.values())
    return max_depth


def build_item_tid_bitsets(transactions, price_lookup):
    """Tạo bitset giao dịch chứa từng item."""
    item_bitsets = {}
    for tid, transaction in enumerate(transactions):
        transaction_bit = 1 << tid
        for item in set(transaction):
            if item in price_lookup:
                item_bitsets[item] = item_bitsets.get(item, 0) | transaction_bit
    return item_bitsets


def build_price_ordered_fp_tree(transactions, frequent_items, price_rank):
    """Xây FP-Tree bằng thứ tự giá tăng dần."""
    frequent_item_set = set(frequent_items)
    tree = PriceOrderedFPTree()
    tree.supports = {}

    for transaction in transactions:
        ordered_items = sorted(
            set(item for item in transaction if item in frequent_item_set),
            key=lambda item: price_rank[item],
        )
        if ordered_items:
            tree.add_transaction(ordered_items)

    return tree


def average_itemset_price(itemset, price_lookup):
    """Tính avg(price) của itemset."""
    return sum(price_lookup[item] for item in itemset) / len(itemset)


def run_proposed_in_mining_pruning(
    transactions,
    item_price,
    item_bitsets,
    min_support_ratio,
    price_threshold=None,
):
    """Chạy Proposed: FP-Tree theo giá và DFS cắt tỉa ngay trong mining."""
    if price_threshold is None:
        price_threshold = PRICE_THRESHOLD

    min_support_count = math.ceil(min_support_ratio * len(transactions))
    price_ordered_items = sorted(item_price, key=lambda item: (item_price[item], item))
    price_rank = {item: rank for rank, item in enumerate(price_ordered_items)}

    start_time = time.perf_counter()

    frequent_items = [
        item
        for item in price_ordered_items
        if item_bitsets.get(item, 0).bit_count() >= min_support_count
    ]

    fp_tree = build_price_ordered_fp_tree(transactions, frequent_items, price_rank)
    all_transaction_bits = (1 << len(transactions)) - 1

    final_patterns = {}
    stats = {
        "root_frequent_items": len(frequent_items),
        "fp_tree_nodes": count_tree_nodes(fp_tree),
        "fp_tree_max_depth": max_tree_depth(fp_tree),
        "recursive_calls": 0,
        "candidate_evaluations": 0,
        "support_evaluations": 0,
        "infrequent_candidates": 0,
        "price_pruned_candidates": 0,
        "max_pattern_length": 0,
    }

    def dfs(start_index, prefix, prefix_price_sum, prefix_tid_bits):
        stats["recursive_calls"] += 1
        stats["max_pattern_length"] = max(stats["max_pattern_length"], len(prefix))

        for item_index in range(start_index, len(frequent_items)):
            item = frequent_items[item_index]
            stats["candidate_evaluations"] += 1

            candidate_price_sum = prefix_price_sum + item_price[item]
            candidate_length = len(prefix) + 1
            candidate_avg_price = candidate_price_sum / candidate_length

            if candidate_avg_price > price_threshold:
                # Vì các item còn lại có giá không nhỏ hơn item hiện tại, toàn bộ phần sau của nhánh đều bị cắt.
                stats["price_pruned_candidates"] += len(frequent_items) - item_index
                break

            stats["support_evaluations"] += 1
            candidate_tid_bits = prefix_tid_bits & item_bitsets[item]
            support_count = candidate_tid_bits.bit_count()

            if support_count >= min_support_count:
                candidate = prefix + (item,)
                canonical_candidate = tuple(sorted(candidate))
                final_patterns[canonical_candidate] = support_count
                dfs(item_index + 1, candidate, candidate_price_sum, candidate_tid_bits)
            else:
                stats["infrequent_candidates"] += 1

    dfs(0, tuple(), 0.0, all_transaction_bits)
    runtime_seconds = time.perf_counter() - start_time

    return {
        "min_support_ratio": min_support_ratio,
        "min_support_label": f"{min_support_ratio:.0%}",
        "min_support_count": min_support_count,
        "runtime_seconds": runtime_seconds,
        "final_patterns": final_patterns,
        "num_final_patterns": len(final_patterns),
        "stats": stats,
    }


index_start_time = time.perf_counter()
item_bitsets = build_item_tid_bitsets(transactions, item_price)
index_build_seconds = time.perf_counter() - index_start_time

print("Đã định nghĩa xong Proposed FP-Tree và DFS in-mining pruning.")
print(f"Thời gian tạo chỉ mục TID bitset dùng chung: {index_build_seconds:.6f} giây")
print("Số item có bitset giao dịch:", len(item_bitsets))

Đã định nghĩa xong Proposed FP-Tree và DFS in-mining pruning.
Thời gian tạo chỉ mục TID bitset dùng chung: 0.609324 giây
Số item có bitset giao dịch: 4006
